# Features Transacciones VPC - TABLA ANCHA
Una sola tabla `HM_TRANSACCIONES_VPC_TOTAL` (panel RUC x periodo) con **todas las categorias** `grupo_final_vpc_n3` en columnas: POS, RECAUDACION, DEPOSITOS, PAGO DE SERVICIOS, PAGOS MASIVOS, PAGOS RECIBIDOS, TRANSFERENCIAS, COBRANZAS, OPERACIONES MESA, REACTIVA.

Por categoria: `monto`, `trxs`, `cashin_monto`, `cashout_monto` (snapshot) + `monto_var1`, `monto_mm6`, `monto_tend6`, `cashin_tend6`. Mas globales (tot, cash-in/out, flujo_neto, n_categorias, meses_activos_6m).

In [ ]:
# 1) Instalacion
%pip install -q awswrangler

In [ ]:
# 2) Imports y CONFIG
import time
import awswrangler as wr

DATABASE       = "disc_comercial"
WORKGROUP      = "ibk-discovery-comercial-work-group"
S3_OUTPUT      = "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/athena-results/"
PERIODO_INICIO = 202401
PERIODO_FIN    = 202605

In [ ]:
# 3) Utilidades
def run_ddl(sql: str, label: str = ""):
    qid = wr.athena.start_query_execution(sql=sql, database=DATABASE, workgroup=WORKGROUP, s3_output=S3_OUTPUT)
    res = wr.athena.wait_query(query_execution_id=qid)
    state = res["Status"]["State"]
    if state != "SUCCEEDED":
        raise RuntimeError("[" + label + "] " + state + ": " + str(res["Status"].get("StateChangeReason","")))
    print("  OK [" + label + "] qid=" + qid)
    return qid

def drop_table(name: str):
    run_ddl("DROP TABLE IF EXISTS " + DATABASE + "." + name, "drop " + name)

In [ ]:
# 4) Template SQL (tabla ancha)
SQL_ANCHA = r"""
-- ============================================================================
-- FEATURES TRANSACCIONES VPC - TABLA ANCHA (todas las categorias en columnas)
-- Fuente : e_perm_aws.t_agg_mes_vpc_transacc_sav_imp
-- Cruce  : e_perm_aws.T_VPC_CLIENTE_BANCA_FINAL_HST (cuc -> ruc)
-- Grano  : num_ruc_val x periodo_val (panel balanceado). 1 fila por RUC x mes.
-- Variables por categoria grupo_final_vpc_n3 (monto/trxs, cash-in/out, tendencias).
-- ============================================================================
-- DROP TABLE IF EXISTS disc_comercial.HM_TRANSACCIONES_VPC_TOTAL

CREATE TABLE disc_comercial.HM_TRANSACCIONES_VPC_TOTAL
WITH (format='Parquet', parquet_compression='SNAPPY', partitioned_by=ARRAY['periodo']) AS (

WITH params AS (
    SELECT {periodo_inicio} AS periodo_inicio, {periodo_fin} AS periodo_fin
),
periodos AS (
    SELECT periodo_val FROM (
        SELECT (anio*100 + mes) AS periodo_val
        FROM (SELECT anio FROM UNNEST(SEQUENCE(2024, 2026)) AS t(anio)) a
        CROSS JOIN (SELECT mes FROM UNNEST(SEQUENCE(1, 12)) AS t(mes)) m
    ) g CROSS JOIN params p
    WHERE g.periodo_val BETWEEN p.periodo_inicio AND p.periodo_fin
),
cuc_ruc AS (
    SELECT periodo_val, num_ruc_val, cuc_num
    FROM e_perm_aws.T_VPC_CLIENTE_BANCA_FINAL_HST
    CROSS JOIN params
    WHERE fecha_dt IN (
        SELECT MAX(fecha_dt) FROM e_perm_aws.T_VPC_CLIENTE_BANCA_FINAL_HST
        CROSS JOIN params p2
        WHERE CAST(YEAR(fecha_dt)*100 + MONTH(fecha_dt) AS INTEGER) BETWEEN p2.periodo_inicio AND p2.periodo_fin
        GROUP BY CAST(YEAR(fecha_dt)*100 + MONTH(fecha_dt) AS INTEGER)
    )
    AND COALESCE(num_ruc_val,'.') NOT LIKE '.' AND COALESCE(num_ruc_val,'.') NOT LIKE ''
),
universo_ruc AS (
    SELECT DISTINCT r.num_ruc_val
    FROM e_perm_aws.t_agg_mes_vpc_transacc_sav_imp t
    INNER JOIN cuc_ruc r ON t.cuc_num_val = r.cuc_num AND t.periodo_val = r.periodo_val
    CROSS JOIN params p
    WHERE t.periodo_val BETWEEN p.periodo_inicio AND p.periodo_fin AND r.num_ruc_val IS NOT NULL
),
malla AS ( SELECT u.num_ruc_val, pe.periodo_val FROM universo_ruc u CROSS JOIN periodos pe ),
base AS (
    SELECT t.periodo_val, r.num_ruc_val,
           t.grupo_final_vpc_n3 AS n3,
           t.tipo_cash_desc     AS flujo,
           t.cant_transacc,
           t.montotransaccion_solar_amt AS monto
    FROM e_perm_aws.t_agg_mes_vpc_transacc_sav_imp t
    INNER JOIN cuc_ruc r ON t.cuc_num_val = r.cuc_num AND t.periodo_val = r.periodo_val
    CROSS JOIN params p
    WHERE t.periodo_val BETWEEN p.periodo_inicio AND p.periodo_fin AND r.num_ruc_val IS NOT NULL
),
agg AS (
    SELECT periodo_val, num_ruc_val,
        SUM(CASE WHEN n3='POS' THEN monto ELSE 0 END)                         AS pos_monto,
        SUM(CASE WHEN n3='POS' THEN cant_transacc ELSE 0 END)                 AS pos_trxs,
        SUM(CASE WHEN n3='POS' AND flujo='CASH-IN'  THEN monto ELSE 0 END)     AS pos_cashin_monto,
        SUM(CASE WHEN n3='POS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)     AS pos_cashout_monto,
        SUM(CASE WHEN n3='RECAUDACION' THEN monto ELSE 0 END)                         AS recaud_monto,
        SUM(CASE WHEN n3='RECAUDACION' THEN cant_transacc ELSE 0 END)                 AS recaud_trxs,
        SUM(CASE WHEN n3='RECAUDACION' AND flujo='CASH-IN'  THEN monto ELSE 0 END)     AS recaud_cashin_monto,
        SUM(CASE WHEN n3='RECAUDACION' AND flujo='CASH-OUT' THEN monto ELSE 0 END)     AS recaud_cashout_monto,
        SUM(CASE WHEN n3='DEPOSITOS' THEN monto ELSE 0 END)                         AS depos_monto,
        SUM(CASE WHEN n3='DEPOSITOS' THEN cant_transacc ELSE 0 END)                 AS depos_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' AND flujo='CASH-IN'  THEN monto ELSE 0 END)     AS depos_cashin_monto,
        SUM(CASE WHEN n3='DEPOSITOS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)     AS depos_cashout_monto,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' THEN monto ELSE 0 END)                         AS pago_serv_monto,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' THEN cant_transacc ELSE 0 END)                 AS pago_serv_trxs,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND flujo='CASH-IN'  THEN monto ELSE 0 END)     AS pago_serv_cashin_monto,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)     AS pago_serv_cashout_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' THEN monto ELSE 0 END)                         AS pagos_masiv_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' THEN cant_transacc ELSE 0 END)                 AS pagos_masiv_trxs,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND flujo='CASH-IN'  THEN monto ELSE 0 END)     AS pagos_masiv_cashin_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)     AS pagos_masiv_cashout_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' THEN monto ELSE 0 END)                         AS pagos_recib_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' THEN cant_transacc ELSE 0 END)                 AS pagos_recib_trxs,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND flujo='CASH-IN'  THEN monto ELSE 0 END)     AS pagos_recib_cashin_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)     AS pagos_recib_cashout_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' THEN monto ELSE 0 END)                         AS transf_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' THEN cant_transacc ELSE 0 END)                 AS transf_trxs,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND flujo='CASH-IN'  THEN monto ELSE 0 END)     AS transf_cashin_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)     AS transf_cashout_monto,
        SUM(CASE WHEN n3='COBRANZAS' THEN monto ELSE 0 END)                         AS cobranz_monto,
        SUM(CASE WHEN n3='COBRANZAS' THEN cant_transacc ELSE 0 END)                 AS cobranz_trxs,
        SUM(CASE WHEN n3='COBRANZAS' AND flujo='CASH-IN'  THEN monto ELSE 0 END)     AS cobranz_cashin_monto,
        SUM(CASE WHEN n3='COBRANZAS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)     AS cobranz_cashout_monto,
        SUM(CASE WHEN n3='OPERACIONES MESA' THEN monto ELSE 0 END)                         AS op_mesa_monto,
        SUM(CASE WHEN n3='OPERACIONES MESA' THEN cant_transacc ELSE 0 END)                 AS op_mesa_trxs,
        SUM(CASE WHEN n3='OPERACIONES MESA' AND flujo='CASH-IN'  THEN monto ELSE 0 END)     AS op_mesa_cashin_monto,
        SUM(CASE WHEN n3='OPERACIONES MESA' AND flujo='CASH-OUT' THEN monto ELSE 0 END)     AS op_mesa_cashout_monto,
        SUM(CASE WHEN n3='REACTIVA' THEN monto ELSE 0 END)                         AS reactiva_monto,
        SUM(CASE WHEN n3='REACTIVA' THEN cant_transacc ELSE 0 END)                 AS reactiva_trxs,
        SUM(CASE WHEN n3='REACTIVA' AND flujo='CASH-IN'  THEN monto ELSE 0 END)     AS reactiva_cashin_monto,
        SUM(CASE WHEN n3='REACTIVA' AND flujo='CASH-OUT' THEN monto ELSE 0 END)     AS reactiva_cashout_monto,
        SUM(monto)                                                              AS tot_monto,
        SUM(cant_transacc)                                                      AS tot_trxs,
        SUM(CASE WHEN flujo='CASH-IN'  THEN monto ELSE 0 END)                    AS cashin_monto_glob,
        SUM(CASE WHEN flujo='CASH-OUT' THEN monto ELSE 0 END)                    AS cashout_monto_glob,
        COUNT(DISTINCT n3)                                                       AS n_categorias
    FROM base GROUP BY periodo_val, num_ruc_val
),
pivot AS (
    SELECT m.num_ruc_val, m.periodo_val,
        COALESCE(a.pos_monto,0) AS pos_monto,
        COALESCE(a.pos_trxs,0) AS pos_trxs,
        COALESCE(a.pos_cashin_monto,0) AS pos_cashin_monto,
        COALESCE(a.pos_cashout_monto,0) AS pos_cashout_monto,
        COALESCE(a.recaud_monto,0) AS recaud_monto,
        COALESCE(a.recaud_trxs,0) AS recaud_trxs,
        COALESCE(a.recaud_cashin_monto,0) AS recaud_cashin_monto,
        COALESCE(a.recaud_cashout_monto,0) AS recaud_cashout_monto,
        COALESCE(a.depos_monto,0) AS depos_monto,
        COALESCE(a.depos_trxs,0) AS depos_trxs,
        COALESCE(a.depos_cashin_monto,0) AS depos_cashin_monto,
        COALESCE(a.depos_cashout_monto,0) AS depos_cashout_monto,
        COALESCE(a.pago_serv_monto,0) AS pago_serv_monto,
        COALESCE(a.pago_serv_trxs,0) AS pago_serv_trxs,
        COALESCE(a.pago_serv_cashin_monto,0) AS pago_serv_cashin_monto,
        COALESCE(a.pago_serv_cashout_monto,0) AS pago_serv_cashout_monto,
        COALESCE(a.pagos_masiv_monto,0) AS pagos_masiv_monto,
        COALESCE(a.pagos_masiv_trxs,0) AS pagos_masiv_trxs,
        COALESCE(a.pagos_masiv_cashin_monto,0) AS pagos_masiv_cashin_monto,
        COALESCE(a.pagos_masiv_cashout_monto,0) AS pagos_masiv_cashout_monto,
        COALESCE(a.pagos_recib_monto,0) AS pagos_recib_monto,
        COALESCE(a.pagos_recib_trxs,0) AS pagos_recib_trxs,
        COALESCE(a.pagos_recib_cashin_monto,0) AS pagos_recib_cashin_monto,
        COALESCE(a.pagos_recib_cashout_monto,0) AS pagos_recib_cashout_monto,
        COALESCE(a.transf_monto,0) AS transf_monto,
        COALESCE(a.transf_trxs,0) AS transf_trxs,
        COALESCE(a.transf_cashin_monto,0) AS transf_cashin_monto,
        COALESCE(a.transf_cashout_monto,0) AS transf_cashout_monto,
        COALESCE(a.cobranz_monto,0) AS cobranz_monto,
        COALESCE(a.cobranz_trxs,0) AS cobranz_trxs,
        COALESCE(a.cobranz_cashin_monto,0) AS cobranz_cashin_monto,
        COALESCE(a.cobranz_cashout_monto,0) AS cobranz_cashout_monto,
        COALESCE(a.op_mesa_monto,0) AS op_mesa_monto,
        COALESCE(a.op_mesa_trxs,0) AS op_mesa_trxs,
        COALESCE(a.op_mesa_cashin_monto,0) AS op_mesa_cashin_monto,
        COALESCE(a.op_mesa_cashout_monto,0) AS op_mesa_cashout_monto,
        COALESCE(a.reactiva_monto,0) AS reactiva_monto,
        COALESCE(a.reactiva_trxs,0) AS reactiva_trxs,
        COALESCE(a.reactiva_cashin_monto,0) AS reactiva_cashin_monto,
        COALESCE(a.reactiva_cashout_monto,0) AS reactiva_cashout_monto,
        COALESCE(a.tot_monto,0) AS tot_monto,
        COALESCE(a.tot_trxs,0) AS tot_trxs,
        COALESCE(a.cashin_monto_glob,0) AS cashin_monto_glob,
        COALESCE(a.cashout_monto_glob,0) AS cashout_monto_glob,
        COALESCE(a.n_categorias,0) AS n_categorias,
        CASE WHEN a.num_ruc_val IS NOT NULL THEN 1 ELSE 0 END AS flag_activo_mes
    FROM malla m
    LEFT JOIN agg a ON m.num_ruc_val=a.num_ruc_val AND m.periodo_val=a.periodo_val
),
panel AS (
    SELECT
        num_ruc_val,
        periodo_val,
        flag_activo_mes,
        pos_monto                                              AS pos_monto_mes,
        pos_cashin_monto                                       AS pos_cashin_mes,
        pos_cashout_monto                                      AS pos_cashout_mes,
        pos_trxs                                               AS pos_trxs_mes,
        pos_monto - LAG(pos_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)               AS pos_monto_var1,
        AVG(CAST(pos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)               AS pos_monto_mm6,
        pos_monto - AVG(CAST(pos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)   AS pos_monto_tend6,
        pos_cashin_monto - AVG(CAST(pos_cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS pos_cashin_tend6,
        recaud_monto                                              AS recaud_monto_mes,
        recaud_cashin_monto                                       AS recaud_cashin_mes,
        recaud_cashout_monto                                      AS recaud_cashout_mes,
        recaud_trxs                                               AS recaud_trxs_mes,
        recaud_monto - LAG(recaud_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)               AS recaud_monto_var1,
        AVG(CAST(recaud_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)               AS recaud_monto_mm6,
        recaud_monto - AVG(CAST(recaud_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)   AS recaud_monto_tend6,
        recaud_cashin_monto - AVG(CAST(recaud_cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS recaud_cashin_tend6,
        depos_monto                                              AS depos_monto_mes,
        depos_cashin_monto                                       AS depos_cashin_mes,
        depos_cashout_monto                                      AS depos_cashout_mes,
        depos_trxs                                               AS depos_trxs_mes,
        depos_monto - LAG(depos_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)               AS depos_monto_var1,
        AVG(CAST(depos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)               AS depos_monto_mm6,
        depos_monto - AVG(CAST(depos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)   AS depos_monto_tend6,
        depos_cashin_monto - AVG(CAST(depos_cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS depos_cashin_tend6,
        pago_serv_monto                                              AS pago_serv_monto_mes,
        pago_serv_cashin_monto                                       AS pago_serv_cashin_mes,
        pago_serv_cashout_monto                                      AS pago_serv_cashout_mes,
        pago_serv_trxs                                               AS pago_serv_trxs_mes,
        pago_serv_monto - LAG(pago_serv_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)               AS pago_serv_monto_var1,
        AVG(CAST(pago_serv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)               AS pago_serv_monto_mm6,
        pago_serv_monto - AVG(CAST(pago_serv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)   AS pago_serv_monto_tend6,
        pago_serv_cashin_monto - AVG(CAST(pago_serv_cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS pago_serv_cashin_tend6,
        pagos_masiv_monto                                              AS pagos_masiv_monto_mes,
        pagos_masiv_cashin_monto                                       AS pagos_masiv_cashin_mes,
        pagos_masiv_cashout_monto                                      AS pagos_masiv_cashout_mes,
        pagos_masiv_trxs                                               AS pagos_masiv_trxs_mes,
        pagos_masiv_monto - LAG(pagos_masiv_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)               AS pagos_masiv_monto_var1,
        AVG(CAST(pagos_masiv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)               AS pagos_masiv_monto_mm6,
        pagos_masiv_monto - AVG(CAST(pagos_masiv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)   AS pagos_masiv_monto_tend6,
        pagos_masiv_cashin_monto - AVG(CAST(pagos_masiv_cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS pagos_masiv_cashin_tend6,
        pagos_recib_monto                                              AS pagos_recib_monto_mes,
        pagos_recib_cashin_monto                                       AS pagos_recib_cashin_mes,
        pagos_recib_cashout_monto                                      AS pagos_recib_cashout_mes,
        pagos_recib_trxs                                               AS pagos_recib_trxs_mes,
        pagos_recib_monto - LAG(pagos_recib_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)               AS pagos_recib_monto_var1,
        AVG(CAST(pagos_recib_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)               AS pagos_recib_monto_mm6,
        pagos_recib_monto - AVG(CAST(pagos_recib_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)   AS pagos_recib_monto_tend6,
        pagos_recib_cashin_monto - AVG(CAST(pagos_recib_cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS pagos_recib_cashin_tend6,
        transf_monto                                              AS transf_monto_mes,
        transf_cashin_monto                                       AS transf_cashin_mes,
        transf_cashout_monto                                      AS transf_cashout_mes,
        transf_trxs                                               AS transf_trxs_mes,
        transf_monto - LAG(transf_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)               AS transf_monto_var1,
        AVG(CAST(transf_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)               AS transf_monto_mm6,
        transf_monto - AVG(CAST(transf_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)   AS transf_monto_tend6,
        transf_cashin_monto - AVG(CAST(transf_cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS transf_cashin_tend6,
        cobranz_monto                                              AS cobranz_monto_mes,
        cobranz_cashin_monto                                       AS cobranz_cashin_mes,
        cobranz_cashout_monto                                      AS cobranz_cashout_mes,
        cobranz_trxs                                               AS cobranz_trxs_mes,
        cobranz_monto - LAG(cobranz_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)               AS cobranz_monto_var1,
        AVG(CAST(cobranz_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)               AS cobranz_monto_mm6,
        cobranz_monto - AVG(CAST(cobranz_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)   AS cobranz_monto_tend6,
        cobranz_cashin_monto - AVG(CAST(cobranz_cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS cobranz_cashin_tend6,
        op_mesa_monto                                              AS op_mesa_monto_mes,
        op_mesa_cashin_monto                                       AS op_mesa_cashin_mes,
        op_mesa_cashout_monto                                      AS op_mesa_cashout_mes,
        op_mesa_trxs                                               AS op_mesa_trxs_mes,
        op_mesa_monto - LAG(op_mesa_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)               AS op_mesa_monto_var1,
        AVG(CAST(op_mesa_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)               AS op_mesa_monto_mm6,
        op_mesa_monto - AVG(CAST(op_mesa_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)   AS op_mesa_monto_tend6,
        op_mesa_cashin_monto - AVG(CAST(op_mesa_cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS op_mesa_cashin_tend6,
        reactiva_monto                                              AS reactiva_monto_mes,
        reactiva_cashin_monto                                       AS reactiva_cashin_mes,
        reactiva_cashout_monto                                      AS reactiva_cashout_mes,
        reactiva_trxs                                               AS reactiva_trxs_mes,
        reactiva_monto - LAG(reactiva_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)               AS reactiva_monto_var1,
        AVG(CAST(reactiva_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)               AS reactiva_monto_mm6,
        reactiva_monto - AVG(CAST(reactiva_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)   AS reactiva_monto_tend6,
        reactiva_cashin_monto - AVG(CAST(reactiva_cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS reactiva_cashin_tend6,
        tot_monto                                              AS tot_monto_mes,
        cashin_monto_glob                                      AS cashin_monto_mes,
        cashout_monto_glob                                     AS cashout_monto_mes,
        cashin_monto_glob - cashout_monto_glob                 AS flujo_neto_mes,
        tot_monto - LAG(tot_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)               AS tot_monto_var1,
        AVG(CAST(tot_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)               AS tot_monto_mm6,
        tot_monto - AVG(CAST(tot_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)   AS tot_monto_tend6,
        cashin_monto_glob - AVG(CAST(cashin_monto_glob AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS cashin_tend6,
        n_categorias                                           AS n_categorias_mes,
        SUM(flag_activo_mes) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                         AS meses_activos_6m
    FROM pivot
)
SELECT p.*, CAST(p.periodo_val AS VARCHAR) AS periodo
FROM panel p
)
"""

## Ejecutar

In [ ]:
# 5) Crear la tabla ancha
t0 = time.time()
drop_table("HM_TRANSACCIONES_VPC_TOTAL")
run_ddl(SQL_ANCHA.format(periodo_inicio=PERIODO_INICIO, periodo_fin=PERIODO_FIN), "tabla ancha")
print("Listo en " + str(round((time.time()-t0)/60,1)) + " min. Tabla: " + DATABASE + ".HM_TRANSACCIONES_VPC_TOTAL")